In [1]:
import numpy as np
import pandas as pd
import sys
import os
import pickle
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import json
sys.path.append("../../src/experiment_util") 
import experiment_utils
import experiment_constants
sys.path.append("../../src/irl") 
import data_corruption_utils
import irl_utils

In [ ]:
with open("./config.json") as f:
    config = json.load(f)
    root_output_dir_path = config["root_output_dir_path"]


In [ ]:
all_user_traj = pd.read_pickle(root_output_dir_path + "/all_user_traj_df.pkl")
sampled_matched_df = pd.read_pickle(root_output_dir_path + "/sampled_matched_perturbed_df.pkl")


In [ ]:
right_subset = all_user_traj[['user', 'traj', 'tp', "traj_count", "traj_total"]]
subset_users = all_user_traj[all_user_traj.user.isin(sampled_matched_df.user.unique())]
rus_user_df = subset_users[subset_users.russian == 1].copy()
non_rus_user_df = subset_users[subset_users.russian == 0].copy()

In [ ]:
# constructing first n df
legality_matrix = experiment_constants.legal_transitions

first_n_df = pd.DataFrame(columns=[
    "user",
    "traj_n",
    "tp_n",
    "traj_counts_n",
    "traj_total_n",
    "run",
    "n",
    "russian"
])

for n in [-1, 50, 40, 30, 25, 20, 15, 10, 5, 3, 1]:
    for run in range(25):
        df_russian_sampled, df_sampled = data_corruption_utils.matched_window(rus_user_df, non_rus_user_df)
        num_s = 12
        num_a = 6
        def first_n(x, n):
            x = x.copy()
            all_traj = x.reshape(-1, 2) 
            if n == -1:
                n_traj = all_traj
            else:
                n_traj = all_traj[:n]
            return data_corruption_utils.reshape_to_n_50_2(n_traj)
         
        def z(x):
            return first_n(x,n)

        df_russian_sampled["traj_n"] = df_russian_sampled["traj"].apply(z)
        df_sampled["traj_n"] = df_sampled["traj_window"].apply(z)

        def tp(x):
            return irl_utils.compute_tp(x.reshape(-1, 2), num_s, num_a)
        df_russian_sampled["tp_n"] = df_russian_sampled["traj_n"].apply(tp)
        df_sampled["tp_n"] = df_sampled["traj_n"].apply(tp)

        def state_count(x):
            return irl_utils.compute_state_count(x.reshape(-1, 2), num_s, num_a)
        df_russian_sampled["traj_counts_n"] = df_russian_sampled["traj_n"].apply(state_count)
        df_sampled["traj_counts_n"] = df_sampled["traj_n"].apply(state_count)

        df_russian_sampled['traj_total_n'] = df_russian_sampled['traj_counts_n'].apply(np.sum)
        df_sampled['traj_total_n'] = df_sampled['traj_counts_n'].apply(np.sum)

        

        for i,row in df_russian_sampled.iterrows():
            new_row = {
                "user":row["user"],
                "traj_n":row["traj_n"],
                "tp_n":row["tp_n"],
                "traj_counts_n":row["traj_counts_n"],
                "traj_total_n":row["traj_total_n"],
                "run":run,
                "n":n,
                "russian":row["russian"],
            }
            first_n_df.loc[len(first_n_df)] = new_row
            
        for i,row in df_sampled.iterrows():
            new_row = {
                "user":row["user"],
                "traj_n":row["traj_n"],
                "tp_n":row["tp_n"],
                "traj_counts_n":row["traj_counts_n"],
                "traj_total_n":row["traj_total_n"],
                "run":run,
                "n":n,
                "russian":row["russian"],
            }
            first_n_df.loc[len(first_n_df)] = new_row
first_n_df["tp_n_legalised"] = first_n_df["tp_n"].apply(lambda x : irl_utils.legalise_tp(x,experiment_constants.legal_transitions) )


In [ ]:

first_n_df.to_pickle(root_output_dir_path + "/first_n_df.pkl")